In [14]:
# Импортируем необходимые библиотеки
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.decomposition import PCA
from umap import UMAP
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, recall_score, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches


In [15]:
# Читаем файл с данными
df = pd.read_csv('dataset_for_CC50_962.csv')

df.head()

,"CC50, mM",MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,175.482382,5.094096,5.094096,0.387225,0.387225,0.417362,42.928571,384.652,340.300,384.350449,...,0,0,0,0,0,0,0,0,3,0
1,5.402819,3.961417,3.961417,0.533868,0.533868,0.462473,45.214286,388.684,340.300,388.381750,...,0,0,0,0,0,0,0,0,3,0
2,161.142320,2.627117,2.627117,0.543231,0.543231,0.260923,42.187500,446.808,388.344,446.458903,...,0,0,0,0,0,0,0,0,3,0
3,107.855654,5.097360,5.097360,0.390603,0.390603,0.377846,41.862069,398.679,352.311,398.366099,...,0,0,0,0,0,0,0,0,4,0
4,139.270991,5.150510,5.150510,0.270476,0.270476,0.429038,36.514286,466.713,424.377,466.334799,...,0,0,0,0,0,0,0,0,0,0


Напишем модель функцию compare_models_linear_regression

In [16]:
def compare_models_linear_regression(df, target_col, test_size=0.2, random_state=42, n_components=15):
    """
    Функция сравнивает несколько моделей регрессии на заданном датасете, выводит результаты и строит графики.

    :param df: dataframe с данными
    :param target_col: название целевого столбца
    :param test_size: доля тестовых данных
    :param random_state: seed для воспроизведения результатов
    :param n_components: количество главных компонент для PCA и UMAP
    """

    # Признаки и целевая переменная
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Конвертация задачи в бинарную для ROC-кривых
    y_bin = (y > y.median()).astype(int)

    # Список моделей
    models = {
        'Linear Regression': LinearRegression(),
        'Decision Tree': DecisionTreeRegressor(random_state=random_state),
        'Random Forest': RandomForestRegressor(random_state=random_state, n_estimators=100),
        'Neural Network': MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=random_state),
        'Support Vector Machine': SVR()
    }

    # Гиперпараметры для Grid Search
    param_grids = {
        'Linear Regression': {},
        'Decision Tree': {'max_depth': [None, 5, 10, 15]},
        'Random Forest': {'n_estimators': [50, 100], 'max_depth': [None, 5, 10]},
        'CatBoost': {'depth': [4, 6, 8], 'learning_rate': [0.01, 0.1]},
        'Neural Network': {'hidden_layer_sizes': [(100,), (100, 50)], 'max_iter': [500, 1000]},
        'Support Vector Machine': {'C': [0.1, 1, 10], 'kernel': ['rbf', 'linear']}
    }

    # Способы уменьшения размерности
    reducers = {
        'Original Features': None,
        'Principal Component Analysis (PCA)': PCA(n_components=n_components, random_state=random_state),
        'Uniform Manifold Approximation and Projection (UMAP)': UMAP(n_components=n_components, random_state=random_state)
    }

    # Сохраняем подготовленные данные
    X_dict, X_train_dict, X_test_dict = {}, {}, {}
    y_train_dict, y_test_dict = {}, {}
    y_bin_train_dict, y_bin_test_dict = {}, {}

    # Подготовка данных для каждого способа уменьшения размерности
    for method, reducer in reducers.items():
        if reducer is None:
            X_red = X
        else:
            X_red = reducer.fit_transform(X)

        # Разделение на тренировочную и тестовую выборки
        X_train_red, X_test_red, y_train_red, y_test_red, y_bin_train_red, y_bin_test_red = train_test_split(
            X_red, y, y_bin, test_size=test_size, random_state=random_state
        )

        # Сохраняем данные
        X_dict[method] = X_red
        X_train_dict[method] = X_train_red
        X_test_dict[method] = X_test_red
        y_train_dict[method] = y_train_red
        y_test_dict[method] = y_test_red
        y_bin_train_dict[method] = y_bin_train_red
        y_bin_test_dict[method] = y_bin_test_red

    # Результаты моделей
    results_all = {key: {} for key in reducers}
    roc_curves_all = {key: {} for key in reducers}
    best_params_all = {key: {} for key in reducers}

    # Подгонка моделей и сбор метрик
    for method in reducers:
        for name, model in models.items():
            param_grid = param_grids[name]

            # Подбираем параметры через GridSearchCV
            gs = GridSearchCV(model, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
            gs.fit(X_train_dict[method], y_train_dict[method])

            # Получаем лучшую модель
            best_model = gs.best_estimator_
            best_params_all[method][name] = gs.best_params_

            # Прогнозы
            y_pred = best_model.predict(X_test_dict[method])
            y_pred_bin = (y_pred > y.median()).astype(int)

            # Собираем метрики
            mse = mean_squared_error(y_test_dict[method], y_pred)
            r2 = r2_score(y_test_dict[method], y_pred)
            acc = accuracy_score(y_bin_test_dict[method], y_pred_bin)
            recall = recall_score(y_bin_test_dict[method], y_pred_bin)
            try:
                roc_auc = roc_auc_score(y_bin_test_dict[method], y_pred)
            except ValueError:
                roc_auc = np.nan

            results_all[method][name] = {
                'MSE': mse,
                'R2': r2,
                'Accuracy': acc,
                'Recall': recall,
                'ROC-AUC': roc_auc
            }

            # Строим ROC-кривую
            fpr, tpr, thresholds = roc_curve(y_bin_test_dict[method], y_pred)
            roc_auc_value = auc(fpr, tpr)
            roc_curves_all[method][name] = (fpr, tpr, roc_auc_value)

    # Вывод лучших параметров и метрик
    metrics = ['Accuracy', 'Recall', 'ROC-AUC', 'R2']
    model_names = list(models.keys())
    reduction_methods = list(reducers.keys())

    for red_method in reduction_methods:
        print(f"\n--- {red_method} ---")
        for model in model_names:
            params = best_params_all[red_method][model]
            res = results_all[red_method][model]
            print(f"{model}:")
            print(f"  Лучшие параметры: {params if params else 'по умолчанию'}")
            print(f"  Accuracy: {res['Accuracy']:.3f}, Recall: {res['Recall']:.3f}, ROC-AUC: {res['ROC-AUC']:.3f}, R2: {res['R2']:.3f}")

    # Горизонтальные столбчатые графики
    fig, axes = plt.subplots(len(metrics), 1, figsize=(23, 7*len(metrics)), sharex=True)
    if len(metrics) == 1:
        axes = [axes]

    colors = ['steelblue', 'darkorange', 'forestgreen']
    bar_height = 0.4

    for i, metric in enumerate(metrics):
        ax = axes[i]
        y_pos = np.arange(len(model_names))

        for j, red_method in enumerate(reduction_methods):
            values = [results_all[red_method][m][metric] for m in model_names]
            bars = ax.barh(
                y_pos + (j-1)*bar_height, values, height=bar_height, color=colors[j], label=red_method, alpha=0.85, edgecolor='black'
            )
            for b in bars:
                width = b.get_width()
                ax.text(width + 0.01, b.get_y() + b.get_height()/2, f"{width:.2f}", va='center', ha='left', fontsize=17, color='black')

        ax.set_yticks(y_pos)
        ax.set_yticklabels(model_names, fontsize=12)
        ax.set_xlabel(metric, fontsize=13)
        ax.set_title(f'Сравнение моделей по метрике: {metric}', fontsize=15, fontweight='bold')
        ax.xaxis.set_major_locator(mticker.MaxNLocator(5))
        ax.grid(axis='x', linestyle='--', alpha=0.5)

    handles = [mpatches.Patch(color=c, label=m) for c, m in zip(colors, reduction_methods)]
    legend_ax = axes[0]
    legend_ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.1), ncol=3, frameon=False, fontsize=12)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    # ROC-кривые
    plt.figure(figsize=(20, 6))
    for j, red_method in enumerate(reduction_methods):
        plt.subplot(1, 3, j+1)
        for name in model_names:
            fpr, tpr, roc_auc_val = roc_curves_all[red_method][name]
            plt.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_val:.2f})', linewidth=2)
        plt.plot([0, 1], [0, 1], 'k--', lw=1)
        plt.title(f'ROC-кривые ({red_method})', fontsize=14, fontweight='bold')
        plt.xlabel('False Positive Rate', fontsize=12)
        plt.ylabel('True Positive Rate', fontsize=12)
        plt.legend(loc='lower right', fontsize=11)
        plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# Вызываем функцию сравнения моделей линейной регрессии
compare_models_linear_regression(
    df,
    target_col='CC50, mM',  # Название целевой переменной
    test_size=0.2,          # Доля тестовых данных
    random_state=42,        # Генерируем воспроизводимые результаты
    n_components=15         # Параметр PCA для снижения размерности
)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
